# Thesis figures — SP vs TS

Every figure is produced from the saved final outputs and written to `V1_Drift/thesis_figures/` as **PNG (300 dpi) + PDF**, with a **CSV** of every plotted number and a **JSON** file recording the configuration and statistics (model, normalization, voxel rule, aggregation, drift statistic, test) — so each figure is traceable as required in THESIS_CURRENT_STATE.

| Figure | Content | Inputs |
|---|---|---|
| 2 | Whole-data validation R², SP vs TS (k1), Henderson et al. as reference | whole-data fits |
| 3 | Single-voxel drift (cvR², Pearson r) | session-wise fits |
| 4 | Population response drift | SP notebook 6, TS `save_perm_pop_TS` (exclAbs100) |
| 5 | RDM drift, all four analyses (1−r / Euclidean × Spearman / RMSE) | same |
| 6 | SP controls, single voxel: matched voxel count, 4 × 6 SF levels, gray background | SP session-wise fits |
| 7 | SP response normalization (raw, zeroMean, equalStd): single voxel, population, RDMs | SP fits and notebook 6 runs |
| S1 | Implementation check against Henderson et al. (2023) | whole-data fits |
| S2 | SP population/RDM drift, all vs matched voxel count | SP notebook 6 (+ matchVox) |

Figures 3–5 follow the poster layout, one row per measure: mean 30 × 30 session matrix in V1 (mean of the 8 subjects) for SP and TS | change from lag 1 per ROI for SP and TS | drift strength per ROI, SP vs TS (mean ± SEM, faint dots = subjects). Fig. 1 of the thesis is the Methods schematic, so numbering starts at 2.

Conventions: SP blue, TS orange in every figure; curves mean ± SEM across subjects; bars mean, dots subjects; drift strength = lag correlation signed so that positive = more drift; stars = session-order permutation test (1000), FDR-BH across the 4 ROIs per model. Missing inputs are skipped and reported (e.g. TS population before `save_perm_pop_TS` has run). Loading the single-voxel results takes several minutes the first time; summaries are cached in `thesis_figures/cache/` (set `RECOMPUTE = True` after new runs).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==== Configuration ====
import os, glob, json, pickle, time, warnings
import numpy as np, h5py, matplotlib as mpl, matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---- inputs (read-only) ----
FITS     = "/content/drive/MyDrive/V2_drift/texturemodel/model_fits/"             # TS fits (whole-data + FAST_CROSS_SESSION)
ROIS_DIR = "/content/drive/MyDrive/V2_drift/nsd/rois/"                            # Henderson ROI info + noise ceilings
NSD      = "/content/drive/MyDrive/V1_Drift/NSD/"                                 # prf-visualrois, pRF R2
SP_SESS  = "/content/drive/MyDrive/V1_Drift/repDrift_expand/"                     # SP session-wise + permPop (+ matchVox/)
SP_WHOLE = SP_SESS + "wholedata/"
SP_SESS_6LEV = "/content/drive/MyDrive/V1_Drift/repDrift_expand_6levels/"              # SP control: 4 ori x 6 SF
SP_SESS_BG   = "/content/drive/MyDrive/V1_Drift/repDrift_expand_with_background/"     # SP control: image on gray background                                                 # SP whole-data fits
TS_POP   = "/content/drive/MyDrive/V2_drift/texturemodel/drift_results/population_permutations/"  # TS permPop
TS_EXCL  = "exclAbs100/"   # TS voxels with |simulated response| > 100% excluded (save_perm_pop_TS EXCL_ABS_PCT); "" = none

# ---- outputs ----
OUT   = "/content/drive/MyDrive/V1_Drift/thesis_figures/"    # PNG (300 dpi) + PDF + CSV (plotted numbers) + JSON (configuration)
CACHE = OUT + "cache/"                                        # computed summaries; delete a file to recompute it

# ---- analysis settings (final configuration) ----
ROIS     = ["V1", "V2", "V3", "hV4"]
CODES    = {"V1": [1, 2], "V2": [3, 4], "V3": [5, 6], "hV4": [7]}
SP_VREG  = {"V1": 1, "V2": 2, "V3": 3, "hV4": 4}
SUBJECTS = list(range(1, 9))
NSESS, NPERM, MATCH_SEED = 30, 1000, 0
TS_TAG   = "k1"                                               # TS configuration: 49 LL + 1 PC per HL group (58 predictors)
SP_POP_ZSTR = ""                                              # normalization used for SP notebook 6 runs ("" = raw)

# ---- style: one colour per model, used in every figure (validated palette, CVD-safe) ----
COL = {"SP": "#257de0", "TS": "#f47230",                       # models
       "Henderson": "#6a4ce2", "TS_herpRF": "#00b894",         # validation figure only
       "SP 6 levels": "#00b894", "SP background": "#6a4ce2", "SP matched voxels": "#d1416f",   # SP-control figure only (SP 4x4 stays blue)
       "SP zeroMean": "#00b894", "SP equalStd": "#6a4ce2"}                                          # normalization figure only
LABEL = {"SP": "SP (4 ori × 4 SF, 18)", "TS": "TS (k1, 58)"}
mpl.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8,
    "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "legend.fontsize": 7.5, "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6, "lines.linewidth": 1.6,
    "axes.edgecolor": "#444444", "xtick.color": "#444444", "ytick.color": "#444444",
    "axes.labelcolor": "#222222", "text.color": "#222222", "savefig.dpi": 300, "pdf.fonttype": 42,
})
W_FULL = 6.3                                                    # inches, A4 text width (16 cm)
os.makedirs(CACHE, exist_ok=True)
print("figures ->", OUT)


In [ ]:
# ==== Shared helpers: statistics, saving, small plotting utilities ====
lag = np.abs(np.subtract.outer(np.arange(NSESS), np.arange(NSESS))); OFF = lag > 0
_rng = np.random.default_rng(0); PERMS = [_rng.permutation(NSESS) for _ in range(NPERM)]

def lag_corr(G):
    """Roth & Merriam: Pearson r between a session x session matrix and session lag (off-diagonal entries)."""
    return np.corrcoef(G[OFF], lag[OFF])[0, 1]

def lag_curve(G):
    return np.array([np.nanmean(G[lag == d]) for d in range(1, NSESS)])

def group_test(Gs, sign):
    """Drift strength per subject (sign * r; positive = more drift), group mean, one-sided permutation p
    (session order permuted identically for all subjects)."""
    subj = np.array([sign * lag_corr(G) for G in Gs])
    null = np.array([np.mean([sign * lag_corr(G[np.ix_(p, p)]) for G in Gs]) for p in PERMS])
    obs = subj.mean()
    return dict(subj=subj, mean=obs, p=(np.sum(null >= obs) + 1) / (len(null) + 1))

def fdr_bh(p):
    p = np.asarray(p, float); n = len(p); o = np.argsort(p)
    q = np.empty(n); q[o] = np.minimum.accumulate((p[o] * n / np.arange(1, n + 1))[::-1])[::-1]
    return np.minimum(q, 1)

def stars(q):
    return "***" if q < 0.001 else "**" if q < 0.01 else "*" if q < 0.05 else "n.s."

def sem(a, axis=0):
    a = np.asarray(a, float); return np.nanstd(a, axis=axis, ddof=1) / np.sqrt(np.sum(np.isfinite(a), axis=axis))

def save_fig(fig, name, meta, rows):
    """PNG (300 dpi) + PDF + CSV of every plotted number + JSON with the full configuration (traceability)."""
    os.makedirs(OUT, exist_ok=True)
    fig.savefig(OUT + name + ".png", bbox_inches="tight"); fig.savefig(OUT + name + ".pdf", bbox_inches="tight")
    if rows:
        keys = list(dict.fromkeys(k for r in rows for k in r))          # union of columns (rows of different panels)
        with open(OUT + name + ".csv", "w") as f:
            f.write(",".join(keys) + "\n")
            for r in rows: f.write(",".join(str(r.get(k, "")) for k in keys) + "\n")
    meta = dict(meta, created=time.strftime("%Y-%m-%d %H:%M"), subjects=SUBJECTS, rois=ROIS)
    with open(OUT + name + ".json", "w") as f: json.dump(meta, f, indent=1, default=str)
    print("saved:", OUT + name + ".{png,pdf,csv,json}")

def bars_with_subjects(ax, xs, values_by_series, colors, labels, width=None, hatch=None):
    """Bar = mean across subjects, error bar = SEM, dots = subjects (no colour-only identity: legend + labels)."""
    k = len(values_by_series); width = width or 0.8 / k
    for j, (vals, c, lab) in enumerate(zip(values_by_series, colors, labels)):
        pos = np.asarray(xs) + (j - (k - 1) / 2) * width
        mu = [np.nanmean(v) if v is not None else np.nan for v in vals]
        er = [sem(v) if v is not None else np.nan for v in vals]
        ax.bar(pos, mu, width * 0.88, color=c, label=lab, edgecolor="white", linewidth=0.8, zorder=2)
        ax.errorbar(pos, mu, yerr=er, fmt="none", ecolor="#222222", elinewidth=0.9, capsize=0, zorder=3)
        for p_, v in zip(pos, vals):
            if v is None: continue
            jit = np.linspace(-width * 0.22, width * 0.22, len(v))
            ax.plot(p_ + jit, v, "o", ms=2.6, mfc="white", mec="#444444", mew=0.6, zorder=4)
    ax.axhline(0, color="#444444", lw=0.6, zorder=1)
    ax.set_xticks(xs); ax.set_xticklabels(ROIS)
    return [np.asarray(xs) + (j - (k - 1) / 2) * width for j in range(k)]

RECOMPUTE = False      # True: ignore cached summaries (e.g. after new runs finished)

def cached(name, fn):
    path = CACHE + name + ".pkl"
    if os.path.exists(path) and not RECOMPUTE:
        with open(path, "rb") as f: print("loaded cache:", path); return pickle.load(f)
    out = fn()
    with open(path, "wb") as f: pickle.dump(out, f)
    return out


In [ ]:
# ==== Data loaders (each result is cached in OUT/cache/) ====
import nibabel as nib

def flat(v): v = v[0] if isinstance(v, (list, tuple)) else v; return np.asarray(v).ravel().astype(int)
def col0(x): x = np.asarray(x); return x[:, 0] if x.ndim > 1 else x

# ---------- whole-data validation R2 (Henderson Fig. 3B voxels: NC > 0.01, no category-ROI overlap) ----------
def _my_fit(s, roi, folder, n_ok):
    for fn in sorted(glob.glob(FITS + f"S{s:02d}/" + folder.format(roi=roi) + "/*/all_fit_params.npy"),
                     key=os.path.getmtime, reverse=True):
        d = np.load(fn, allow_pickle=True).item()
        if n_ok(np.asarray(d["best_params"][1]).shape[1]): return dict(zip(flat(d["voxel_index"]), col0(d["val_r2"])))
def _her_fit(s, roi):
    fn = FITS + f"MAGGIES/S{s:02d}/all_fit_params.npy"
    if os.path.exists(fn):
        d = np.load(fn, allow_pickle=True).item(); return dict(zip(flat(d["voxel_index"]), col0(d["val_r2"])))
def _sp_fit(s, roi):
    fn = SP_WHOLE + f"SP_wholedata_sub{s}.npy"
    if os.path.exists(fn):
        d = np.load(fn, allow_pickle=True).item()[roi]; return dict(zip(d["voxel_index"], d["val_r2"]))
WHOLE_MODELS = {
    "Henderson et al. (2023), TS":      _her_fit,
    "TS, Henderson pRFs (validation)":  lambda s, r: _my_fit(s, r, "texture_pyramid_OLS_4ori_4sf_pcaHL_HendersonGroups_{roi}_pca95_use_henderson_grid5_pRFs", lambda n: n > 60),
    "TS k1 (58), NSD pRFs":             lambda s, r: _my_fit(s, r, "texture_pyramid_OLS_4ori_4sf_pcaHL_HendersonGroups_{roi}_k1_use_nsd_grid5_pRFs", lambda n: n == 58),
    "TS LL only (49), NSD pRFs":        lambda s, r: _my_fit(s, r, "texture_pyramid_OLS_4ori_4sf_pcaHL_HendersonGroups_{roi}_k1_LLonly_use_nsd_grid5_pRFs", lambda n: n == 49),
    "SP (18), NSD pRFs":                _sp_fit,
}
def load_wholedata():
    out = {m: np.full((len(SUBJECTS), 4), np.nan) for m in WHOLE_MODELS}
    for si, s in enumerate(SUBJECTS):
        info = np.load(ROIS_DIR + f"S{s}_voxel_roi_info.npy", allow_pickle=True).item()
        vidx = flat(info["voxel_idx"]); nc = np.load(ROIS_DIR + f"S{s}_noise_ceiling_avgreps.npy") / 100
        ret, face, place, body = (np.asarray(info[k])[vidx] for k in ["roi_labels_retino", "roi_labels_face", "roi_labels_place", "roi_labels_body"])
        categ = np.isin(face, [2, 3]) | np.isin(place, [1, 2, 3]) | np.isin(body, [1])
        for ri, roi in enumerate(ROIS):
            sel = vidx[np.isin(ret, CODES[roi]) & ~categ & (nc > 0.01)]
            for m, load in WHOLE_MODELS.items():
                lk = load(s, roi)
                if lk is None: continue
                v = np.array([lk.get(x, np.nan) for x in sel], float)
                if not m.startswith("Henderson"): v[v < -1] = np.nan      # OLS-unstable voxels excluded (counted in the log)
                out[m][si, ri] = np.nanmean(v)
    return out

# ---------- single-voxel cross-session matrices (Roth & Merriam: pRF R2 > 0 voxels, median per session pair) ----------
def voxel_sets(s, match):
    ld = lambda f: nib.load(NSD + f"subj{s:02d}/func1pt8mm/{f}").get_fdata().ravel()
    vis, pr2 = ld("roi/prf-visualrois.nii.gz"), ld("R2.nii.gz")
    goods = {roi: np.isin(vis, CODES[roi]) & (pr2 > 0) for roi in ROIS}
    if match:
        n_min = min(int(g.sum()) for g in goods.values())
        for ri, roi in enumerate(ROIS):
            idx = np.random.default_rng([MATCH_SEED, s, ri]).choice(np.flatnonzero(goods[roi]), n_min, replace=False)
            goods[roi] = np.zeros_like(goods[roi]); goods[roi][idx] = True
    return goods

def _ts_sv(s, roi, good):
    fns = glob.glob(FITS + f"S{s:02d}/FAST_CROSS_SESSION/S{s:02d}_{roi}_texture_pyramid_OLS_4ori_4sf_"
                    f"pcaHL_HendersonGroups_{roi}_{TS_TAG}_use_nsd_grid5_pRFs_*cross_session.h5py")
    if not fns: return None
    with h5py.File(fns[0], "r") as f:
        keep = good[np.asarray(f["voxel_index"]).ravel()]
        return {k: np.nanmedian(np.asarray(f[k][:NSESS, :NSESS, :, 0])[:, :, keep], axis=2) for k in ["val_r2", "val_cc"]}

def _sp_sv(s, roi, good, z, folder=None):
    fn = (folder or SP_SESS) + f"regressPrfSplit_session_v{SP_VREG[roi]}_sub{s}{z}.pkl"
    if not os.path.exists(fn): return None
    with open(fn, "rb") as f: D = pickle.load(f)
    nsd = D["nsd"]; vi = np.concatenate([np.asarray(nsd["roiInd"][rn]).ravel() for rn in D["rois"]]); keep = good[vi]
    out = {}
    for key, name in [("r2oriSplit", "val_r2"), ("pearsonRori", "val_cc")]:
        M = np.concatenate([np.asarray(nsd[key][rn]) for rn in D["rois"]], axis=1)    # [data sess, vox, model sess]
        out[name] = np.nanmedian(M[:NSESS, keep, :NSESS], axis=1).T                  # -> [train, test]
    return out

SV_SOURCES = {   # all raw betas; SP controls only in Figure 5
    "SP": lambda s, r, g: _sp_sv(s, r, g, ""),
    "TS": _ts_sv,
    "SP 6 levels": lambda s, r, g: _sp_sv(s, r, g, "", SP_SESS_6LEV),
    "SP background": lambda s, r, g: _sp_sv(s, r, g, "", SP_SESS_BG),
    "SP zeroMean": lambda s, r, g: _sp_sv(s, r, g, "_zeroMean"),          # normalization figure (SP only)
    "SP equalStd": lambda s, r, g: _sp_sv(s, r, g, "_equalStd"),
}

def load_single_voxel(match):
    res = {}
    for s in SUBJECTS:
        goods = voxel_sets(s, match); print(f"single voxel, S{s}, matched={match}")
        for roi in ROIS:
            for m, fn in SV_SOURCES.items():
                d = fn(s, roi, goods[roi])
                if d is None: continue
                for k, v in d.items(): res.setdefault((m, roi, k), []).append(v)
    return {k: np.array(v) for k, v in res.items()}

# ---------- population / RDM drift (outputs of SP notebook 6 and TS save_perm_pop_TS) ----------
POP_MEASURES = {   # name: (file metric tag, matrix key, permutation key, similarity?)
    "Population response similarity": ("rdmCorrDist_cmpSpearman", "avgImgCorrMat", "imgCorrMatPerm", True),
    "RDM (1−r), Spearman":            ("rdmCorrDist_cmpSpearman", "betweenSessCorr", "betweenSessCorrPerm", True),
    "RDM (Euclidean), RMSE":          ("rdmEuclidean_cmpRMSE", "betweenSessCorr", "betweenSessCorrPerm", False),
    "RDM (1−r), RMSE":                ("rdmCorrDist_cmpRMSE", "betweenSessCorr", "betweenSessCorrPerm", False),
    "RDM (Euclidean), Spearman":      ("rdmEuclidean_cmpSpearman", "betweenSessCorr", "betweenSessCorrPerm", True),
}
def _pop_file(model, s, roi, tag, match, zstr=None):
    if model == "SP":
        z = SP_POP_ZSTR if zstr is None else zstr
        return SP_SESS + ("matchVox/" if match else "") + f"permPop_N{NPERM}_v{SP_VREG[roi]}_sub{s}{z}_{tag}.npz", "Ori"
    return TS_POP + TS_EXCL + ("matchVox/" if match else "") + f"permPop_TS_{TS_TAG}_raw_N{NPERM}_S{s:02d}_{roi}_{tag}.npz", ""

def load_population(match, models=("SP", "TS"), sp_zstr=None, label=None):
    """Per model x measure x ROI: per-subject matrices (no permutations needed: the group test re-permutes).
    The matched-voxel-count control is run for SP only: load_population(True, models=("SP",))."""
    res, missing = {}, []
    for model in models:
        for meas, (tag, key, pkey, is_sim) in POP_MEASURES.items():
            for roi in ROIS:
                Ms = []
                for s in SUBJECTS:
                    fn, suf = _pop_file(model, s, roi, tag, match, sp_zstr)
                    if not os.path.exists(fn): missing.append(os.path.basename(fn)); continue
                    with np.load(fn, allow_pickle=True) as d: Ms.append(np.asarray(d[key + suf], float))
                if Ms: res[(label or model, meas, roi)] = np.array(Ms)
    if missing: print(f"population, matched={match}: {len(missing)} files missing, e.g. {missing[:3]}")
    return res


In [ ]:
# ==== Figure functions ====
from matplotlib.ticker import MaxNLocator

def _fig_legend(fig, models, colors, labels, y=1.0):
    handles = [plt.Line2D([], [], color=colors[m], lw=2.2) for m in models]
    fig.legend(handles, [labels[m] for m in models], loc="lower center", ncol=len(models),
               bbox_to_anchor=(0.5, y), fontsize=7.5, handlelength=1.8, columnspacing=1.6)

def _lag_axes(ax, title=None, ylabel=None):
    ax.set_xlim(0.5, NSESS - 0.5); ax.set_xticks([1, 10, 20, 29])
    ax.yaxis.set_major_locator(MaxNLocator(4))
    if title: ax.set_title(title)
    if ylabel: ax.set_ylabel(ylabel)

def _curve(ax, Gs, color, label):
    C = np.array([lag_curve(G) for G in Gs]); x = np.arange(1, NSESS)
    mu, se = np.nanmean(C, 0), sem(C)
    ax.fill_between(x, mu - se, mu + se, color=color, alpha=0.18, lw=0)
    ax.plot(x, mu, color=color, label=label)
    return mu, se

def _panel_label(ax, s):
    ax.text(-0.16, 1.06, s, transform=ax.transAxes, fontsize=10, fontweight="bold", va="bottom")

def _sig_text(ax, x, y, txt):
    ax.text(x, y, txt, ha="center", va="bottom", fontsize=7, color="#222222")

# ---------- Figure 2: whole-data validation R2, SP vs TS ----------
def figure1(W):
    fig, ax = plt.subplots(figsize=(W_FULL * 0.62, 2.6))
    series = [("SP (18), NSD pRFs", COL["SP"], LABEL["SP"]), ("TS k1 (58), NSD pRFs", COL["TS"], LABEL["TS"])]
    xs = np.arange(4)
    pos = bars_with_subjects(ax, xs, [[W[m][:, ri] for ri in range(4)] for m, _, _ in series],
                             [c for _, c, _ in series], [l for _, _, l in series])
    her = W.get("Henderson et al. (2023), TS")
    if her is not None and np.isfinite(her).any():                          # published reference, same voxels
        for ri in range(4):
            ax.plot([xs[ri] - 0.38, xs[ri] + 0.38], [np.nanmean(her[:, ri])] * 2, color="#222222", lw=1.0, ls=(0, (3, 2)), zorder=5)
        ax.plot([], [], color="#222222", lw=1.0, ls=(0, (3, 2)), label="Henderson et al. (2023), TS")
    rows, ps = [], []
    for ri, roi in enumerate(ROIS):
        a, b = W["TS k1 (58), NSD pRFs"][:, ri], W["SP (18), NSD pRFs"][:, ri]; ok = np.isfinite(a) & np.isfinite(b)
        from scipy.stats import wilcoxon
        p = wilcoxon(a[ok] - b[ok]).pvalue if ok.sum() > 5 else np.nan; ps.append(p)
        for m in W: rows.append(dict(model=m, roi=roi, mean=np.nanmean(W[m][:, ri]), sem=sem(W[m][:, ri]),
                                     **{f"S{s}": W[m][si, ri] for si, s in enumerate(SUBJECTS)}))
    q = fdr_bh(ps)
    top = np.nanmax([W[m] for m in W if not m.startswith("TS, H")]) * 1.06
    for ri in range(4): _sig_text(ax, xs[ri], top, stars(q[ri]) if np.isfinite(q[ri]) else "")
    ax.set_ylabel("validation R² (whole-data fit)"); ax.set_ylim(0, top * 1.12)
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2, fontsize=7)
    meta = dict(figure="Whole-data validation R2, SP vs TS",
                voxels="Henderson Fig. 3B rule: ROI voxels with noise ceiling > 0.01, category-ROI overlap removed; OLS voxels with R2 < -1 excluded",
                fit="all sessions, averaged repetitions, validation on shared-1000 images, OLS, NSD pRFs (TS: aligned to grid 5)",
                test="paired Wilcoxon TS vs SP per ROI across subjects, FDR-BH over 4 ROIs", q_TS_vs_SP=dict(zip(ROIS, q)))
    save_fig(fig, "Fig2_wholedata_SP_vs_TS", meta, rows); plt.show()
    print("Caption draft: Validation R² of whole-data fits for the SP (blue) and TS k1 (orange) models with NSD pRFs. "
          "Bars, mean across subjects; error bars, SEM; dots, subjects; dashed lines, Henderson et al. (2023) on the same voxels. "
          "Stars, paired Wilcoxon TS vs SP, FDR-corrected across ROIs.")

def figure1_validation(W):
    """Supplementary: implementation check against Henderson et al. (2023)."""
    fig, ax = plt.subplots(figsize=(W_FULL * 0.75, 2.6))
    series = [("Henderson et al. (2023), TS", COL["Henderson"], "Henderson et al. (2023), her pRFs"),
              ("TS, Henderson pRFs (validation)", COL["TS_herpRF"], "This study, TS, Henderson pRFs"),
              ("TS k1 (58), NSD pRFs", COL["TS"], "This study, TS k1, NSD pRFs"),
              ("SP (18), NSD pRFs", COL["SP"], "This study, SP, NSD pRFs")]
    bars_with_subjects(ax, np.arange(4), [[W[m][:, ri] for ri in range(4)] for m, _, _ in series],
                       [c for _, c, _ in series], [l for _, _, l in series], width=0.2)
    ax.set_ylabel("validation R² (whole-data fit)"); ax.legend(fontsize=6.8, loc="upper right")
    rows = [dict(model=m, roi=roi, mean=np.nanmean(W[m][:, ri]), sem=sem(W[m][:, ri])) for m in W for ri, roi in enumerate(ROIS)]
    save_fig(fig, "FigS1_validation_Henderson", dict(figure="Implementation check against Henderson et al. (2023)"), rows); plt.show()


# ==================== Poster-layout figures (thesis Figs 3-7) ====================
# One row per measure:
#   [mean 30x30 session matrix, V1, SP] [same, TS] [change from lag 1, SP, by ROI] [same, TS] [drift strength across ROIs, SP vs TS]
ROI_COL = {"V1": "#1b1340", "V2": "#4a3a9a", "V3": "#7d6cd0", "hV4": "#b0a4e6"}   # ordered ROIs: one hue, dark -> light

def _drift_stats(get, models, key, sign):
    """Group permutation test per model x ROI, FDR-BH across the 4 ROIs within each model."""
    st = {}
    for m in models:
        res = {roi: group_test(get(m, key, roi), sign) for roi in ROIS if get(m, key, roi) is not None}
        q = fdr_bh([res[r]["p"] for r in res]) if res else []
        for (roi, r), qq in zip(res.items(), q): st[(m, roi)] = dict(r, q=qq)
    return st

def _matrix_panel(ax, Gs, title, cbar_label, axis_labels=True):
    M = np.nanmean(Gs, 0).astype(float); np.fill_diagonal(M, np.nan)
    lo, hi = np.nanpercentile(M, [2, 98])
    im = ax.imshow(M, cmap="viridis", vmin=lo, vmax=hi, interpolation="nearest")
    ax.set_xticks([0, 29]); ax.set_xticklabels([1, 30]); ax.set_yticks([0, 29]); ax.set_yticklabels([1, 30])
    if axis_labels: ax.set_xlabel("session", labelpad=1); ax.set_ylabel("session", labelpad=1)
    ax.set_title(f"{title}\n{cbar_label}", fontsize=7, linespacing=1.15)
    for sp_ in ax.spines.values(): sp_.set_visible(False)
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.03); cb.ax.tick_params(labelsize=5.5, width=0.5, length=2)
    cb.set_ticks([lo, hi]); cb.ax.set_yticklabels([f"{lo:.2g}", f"{hi:.2g}"]); cb.outline.set_linewidth(0.4)
    return M

def _delta_panel(ax, get, model, key, title, ylabel):
    rows = []
    for roi in ROIS:
        Gs = get(model, key, roi)
        if Gs is None: continue
        C = np.array([lag_curve(G) for G in Gs]); C = C - C[:, :1]          # change from lag 1, per subject
        mu, se = np.nanmean(C, 0), sem(C); x = np.arange(1, NSESS)
        ax.fill_between(x, mu - se, mu + se, color=ROI_COL[roi], alpha=0.15, lw=0)
        ax.plot(x, mu, color=ROI_COL[roi], lw=1.3, label=roi)
        rows += [dict(model=model, measure=key, roi=roi, lag=int(l), delta_mean=m_, delta_sem=s_) for l, m_, s_ in zip(x, mu, se)]
    ax.axhline(0, color="#999999", lw=0.5, zorder=0)
    _lag_axes(ax, title=title, ylabel=ylabel); ax.set_xlabel("session lag", labelpad=1)
    return rows

def _roi_compare_panel(ax, st, series, colors, labels, ylabel="drift strength", star_y=None):
    """Poster style: mean ± SEM per ROI, connected across ROIs; faint dots = subjects; stars = FDR-corrected group test."""
    k = len(series); off = np.linspace(-0.12, 0.12, k) if k > 1 else [0.0]; xs = np.arange(len(ROIS))
    rows, tops = [], []
    for j, m in enumerate(series):
        mu = np.array([st[(m, r)]["mean"] if (m, r) in st else np.nan for r in ROIS])
        se = np.array([sem(st[(m, r)]["subj"]) if (m, r) in st else np.nan for r in ROIS])
        for i, r in enumerate(ROIS):
            if (m, r) not in st: continue
            v = st[(m, r)]["subj"]
            ax.plot(np.full(len(v), xs[i] + off[j]), v, "o", ms=2.2, color=colors[m], alpha=0.35, mew=0, zorder=2)
            tops.append(np.nanmax(v))
            rows.append(dict(model=m, roi=r, drift_mean=st[(m, r)]["mean"], sem=sem(v), p=st[(m, r)]["p"], q_fdr=st[(m, r)]["q"], n=len(v),
                             **{f"S{SUBJECTS[si]}": vv for si, vv in enumerate(v)}))
        ax.errorbar(xs + off[j], mu, yerr=se, color=colors[m], lw=1.4, marker="o", ms=4, capsize=0, elinewidth=1.1,
                    label=labels.get(m, m), zorder=3)
    rng_ = ax.get_ylim(); pad = 0.03 * (rng_[1] - rng_[0]); top = max(tops) if tops else 0
    for j, m in enumerate(series):
        for i, r in enumerate(ROIS):
            if (m, r) in st and st[(m, r)]["q"] < 0.05:
                y = st[(m, r)]["mean"] + sem(st[(m, r)]["subj"]) + pad
                ax.text(xs[i] + off[j], y, stars(st[(m, r)]["q"]), ha="center", va="bottom", fontsize=5.5,
                        color="#222222")
    ax.axhline(0, color="#999999", lw=0.5, zorder=0)
    ax.set_xticks(xs); ax.set_xticklabels(ROIS); ax.set_xlim(-0.5, len(ROIS) - 0.5)
    ax.yaxis.set_major_locator(MaxNLocator(4)); ax.set_ylabel(ylabel)
    lo, hi = ax.get_ylim(); ax.set_ylim(lo, max(hi, top + 2 * pad))
    return rows

def figure_rows(name, rows_spec, get, caption, models=("SP", "TS"), colors=None, labels=None, meta_extra=None):
    """rows_spec: list of dict(title, key, sim (bool), mat_label, delta_label)."""
    colors = colors or COL; labels = labels or LABEL
    n = len(rows_spec); H = 1.8 * n + 0.6; fig = plt.figure(figsize=(W_FULL * 1.25, H))
    gs = fig.add_gridspec(n, 8, width_ratios=[1, 0.22, 1, 0.62, 1.1, 1.1, 0.12, 1.3], wspace=0.5, hspace=0.85,
                          top=1 - 0.75 / H, bottom=0.35 / H)
    COLS = [0, 2, 4, 5, 7]                                     # columns 1, 3 and 6 are spacers
    all_rows, stats_out = [], {}
    for i, spec in enumerate(rows_spec):
        sign = -1.0 if spec["sim"] else 1.0
        st = _drift_stats(get, models, spec["key"], sign); stats_out[spec["title"]] = st
        axes = [fig.add_subplot(gs[i, j]) for j in COLS]
        for j, m in enumerate(models[:2]):
            Gs = get(m, spec["key"], "V1")
            if Gs is not None: _matrix_panel(axes[j], Gs, f"{m}, V1 (mean of {len(Gs)})", spec["mat_label"], axis_labels=(j == 0))
            else: axes[j].axis("off")
        shared = []
        for j, m in enumerate(models[:2]):
            all_rows += _delta_panel(axes[2 + j], get, m, spec["key"], m, spec["delta_label"] if j == 0 else None)
            shared.append(axes[2 + j])
        if len(shared) == 2:                                   # same y-scale for the two models' change-from-lag-1 panels
            lo = min(a.get_ylim()[0] for a in shared); hi = max(a.get_ylim()[1] for a in shared)
            for a in shared: a.set_ylim(lo, hi)
            shared[1].tick_params(labelleft=False)
        all_rows += [dict(r, measure=spec["title"]) for r in _roi_compare_panel(axes[4], st, list(models), colors, labels)]
        axes[0].text(-0.45, 1.42, spec["title"], transform=axes[0].transAxes, fontsize=8.5, fontweight="bold", va="bottom")
    h_roi = [plt.Line2D([], [], color=ROI_COL[r], lw=1.6) for r in ROIS]
    h_mod = [plt.Line2D([], [], color=colors[m], lw=1.6, marker="o", ms=4) for m in models]
    fig.legend(h_mod + h_roi, [labels.get(m, m) for m in models] + ROIS, loc="lower center", ncol=len(models) + 4,
               bbox_to_anchor=(0.5, 1 - 0.3 / H), fontsize=7)
    meta = dict(figure=name, layout="per row: mean session matrix V1 (SP, TS) | change from lag 1 by ROI (SP, TS) | drift strength per ROI",
                drift="Pearson r(metric, session lag) over off-diagonal entries, sign so that positive = more drift; mean over subjects",
                test="group permutation test, 1000 session orders identical for all subjects, one-sided; FDR-BH over 4 ROIs per model",
                stats={t: {f"{m} {r}": dict(mean=v["mean"], p=v["p"], q=v["q"]) for (m, r), v in st.items()} for t, st in stats_out.items()},
                **(meta_extra or {}))
    save_fig(fig, name, meta, all_rows); plt.show(); print("Caption draft:", caption)
    return stats_out

def figure_series_grid(name, panels, series, colors, labels, caption, ncol=3, meta_extra=None):
    """Drift strength per ROI for several series (conditions) side by side. panels: list of (title, get, key, sim)."""
    nrow = int(np.ceil(len(panels) / ncol)); fig, axs = plt.subplots(nrow, ncol, figsize=(W_FULL, 2.0 * nrow + 0.3), squeeze=False)
    all_rows, stats_out = [], {}
    for ax, (title, get, key, sim) in zip(axs.ravel(), panels):
        st = _drift_stats(get, series, key, -1.0 if sim else 1.0); stats_out[title] = st
        all_rows += [dict(r, measure=title) for r in _roi_compare_panel(ax, st, series, colors, labels)]
        ax.set_title(title, fontsize=8)
    for ax in axs.ravel()[len(panels):]: ax.axis("off")
    for ax in axs[:, 1:].ravel(): ax.set_ylabel("")
    h = [plt.Line2D([], [], color=colors[m], lw=1.6, marker="o", ms=4) for m in series]
    fig.legend(h, [labels.get(m, m) for m in series], loc="lower center", ncol=len(series), bbox_to_anchor=(0.5, 1.0))
    plt.tight_layout()
    meta = dict(figure=name, test="group permutation test (1000 session orders), FDR-BH over 4 ROIs per series",
                stats={t: {f"{m} {r}": dict(mean=v["mean"], p=v["p"], q=v["q"]) for (m, r), v in st.items()} for t, st in stats_out.items()},
                **(meta_extra or {}))
    save_fig(fig, name, meta, all_rows); plt.show(); print("Caption draft:", caption)
    return stats_out

# ---- data accessors ----
def sv_get(SV):   return lambda m, key, roi: SV.get((m, roi, key))
def pop_get(P):   return lambda m, key, roi: P.get((m, key, roi))

SV_ROWS = [dict(title="Cross-session cvR²", key="val_r2", sim=True, mat_label="median cvR²", delta_label="Δ cvR² from lag 1"),
           dict(title="Cross-session Pearson r", key="val_cc", sim=True, mat_label="median r", delta_label="Δ r from lag 1")]
POP_ROWS = [dict(title="Population response similarity", key="Population response similarity", sim=True,
                 mat_label="similarity (r)", delta_label="Δ similarity from lag 1")]
RDM_ROWS = [dict(title="RDM 1−r, compared with Spearman", key="RDM (1−r), Spearman", sim=True, mat_label="Spearman ρ", delta_label="Δ ρ from lag 1"),
            dict(title="RDM Euclidean, compared with Spearman", key="RDM (Euclidean), Spearman", sim=True, mat_label="Spearman ρ", delta_label="Δ ρ from lag 1"),
            dict(title="RDM 1−r, compared with RMSE", key="RDM (1−r), RMSE", sim=False, mat_label="RMSE", delta_label="Δ RMSE from lag 1"),
            dict(title="RDM Euclidean, compared with RMSE", key="RDM (Euclidean), RMSE", sim=False, mat_label="RMSE", delta_label="Δ RMSE from lag 1")]

def fig3_single_voxel(SV):
    return figure_rows("Fig3_single_voxel_SP_vs_TS", SV_ROWS, sv_get(SV),
        "Single-voxel drift. Rows: cross-session cvR² and Pearson r. (A, B) Mean train × test session matrix in V1 "
        "(median over voxels with NSD pRF R² > 0, mean over 8 subjects). (C, D) Change from lag 1 as a function of session lag, "
        "per ROI (mean ± SEM across subjects). (E) Drift strength per ROI (correlation with lag, positive = more drift); "
        "points, mean ± SEM; faint dots, subjects; stars, group permutation test, FDR-corrected across ROIs.",
        meta_extra=dict(voxels="all ROI voxels with NSD pRF R2 > 0; median across voxels per session pair", betas="raw (single trials)"))

def fig4_population(P):
    return figure_rows("Fig4_population_SP_vs_TS", POP_ROWS, pop_get(P),
        "Population response drift. (A, B) Mean session × session population response similarity in V1 (correlation across voxels of "
        "the simulated responses to the same image, averaged over 100 images; mean over 8 subjects). (C, D) Change from lag 1 per ROI. "
        "(E) Drift strength per ROI, SP vs TS. TS: voxels with simulated responses > 100% signal change excluded.",
        meta_extra=dict(voxels="all voxels; TS: |simulated response| > 100% excluded (exclAbs100)", images="100 simulated images"))

def fig5_rdm(P):
    return figure_rows("Fig5_RDM_SP_vs_TS", RDM_ROWS, pop_get(P),
        "Representational geometry drift. Rows: the four RDM analyses (RDMs from 1−r or Euclidean distance; compared across sessions "
        "with Spearman correlation or RMSE). Columns as in Fig. 4. For RMSE, larger values mean more different RDMs, so drift is an increase.",
        meta_extra=dict(voxels="all voxels; TS: exclAbs100"))

def fig6_sp_controls(SV, SVm):
    """SP controls (single voxel): matched voxel count, 4 x 6 SF levels, images on a gray background."""
    D = dict(SV)
    for roi in ROIS:
        for k in ("val_r2", "val_cc"):
            if ("SP", roi, k) in SVm: D[("SP matched voxels", roi, k)] = SVm[("SP", roi, k)]
    series = [m for m in ["SP", "SP matched voxels", "SP 6 levels", "SP background"] if any((m, r, "val_r2") in D for r in ROIS)]
    n = {m: max(len(D[(m, r, "val_r2")]) for r in ROIS if (m, r, "val_r2") in D) for m in series}
    labels = {"SP": "SP 4 × 4 (18)", "SP matched voxels": "SP 4 × 4, matched voxel count", "SP 6 levels": "SP 4 × 6 (26)",
              "SP background": "SP, gray background"}
    labels = {m: f"{labels[m]} (n={n[m]})" for m in series}
    get = sv_get(D)
    return figure_series_grid("Fig6_SP_controls", [("Cross-session cvR²", get, "val_r2", True), ("Cross-session Pearson r", get, "val_cc", True)],
        series, COL, labels, "Single-voxel drift strength for the SP model and its controls: matched voxel count (every ROI subsampled to the "
        "subject's smallest ROI), six spatial-frequency levels, and images on a gray background. Mean ± SEM; faint dots, subjects; "
        "stars, FDR-corrected group permutation test.", ncol=2)

def fig7_normalization(SVn, Pn):
    """SP response normalization (raw, zeroMean, equalStd): single voxel, population and the four RDM analyses."""
    series = ["SP", "SP zeroMean", "SP equalStd"]
    labels = {"SP": "raw", "SP zeroMean": "session mean removed (zeroMean)", "SP equalStd": "variance equalized (equalStd)"}
    gp = pop_get(Pn)
    panels = [("Single voxel: cvR²", sv_get(SVn), "val_r2", True), ("Population similarity", gp, "Population response similarity", True)] + \
             [(r["title"].replace(", compared with", " /"), gp, r["key"], r["sim"]) for r in RDM_ROWS]
    return figure_series_grid("Fig7_SP_normalization", panels, series, COL, labels,
        "Drift strength under response normalization (SP model). Removing each voxel's session mean abolishes single-voxel and population "
        "drift and the rank-based 1−r RDM drift; Euclidean RDMs are unchanged by mean removal by construction. Mean ± SEM; faint dots, "
        "subjects; stars, FDR-corrected group permutation test.", ncol=3)

def figS2_sp_matched(P, Pm):
    """Supplementary: SP population and RDM drift, all voxels vs matched voxel count."""
    D = dict(P)
    for (m, meas, roi), v in Pm.items():
        if m == "SP": D[("SP matched voxels", meas, roi)] = v
    gp = pop_get(D); series = ["SP", "SP matched voxels"]
    labels = {"SP": "all voxels", "SP matched voxels": "matched voxel count"}
    panels = [("Population similarity", gp, "Population response similarity", True)] + \
             [(r["title"].replace(", compared with", " /"), gp, r["key"], r["sim"]) for r in RDM_ROWS]
    return figure_series_grid("FigS2_SP_matched_population_RDM", panels, series, COL, labels,
        "SP population and RDM drift with all voxels and with every ROI subsampled to the subject's smallest ROI (one seeded draw).", ncol=3)


## Figure 2 — whole-data validation R², SP vs TS (+ Fig. S1, implementation check)

In [ ]:
W = cached("wholedata", load_wholedata)
figure1(W)
figure1_validation(W)

## Figure 3 — single-voxel drift, SP vs TS

In [ ]:
SV  = cached("single_voxel_all_v2", lambda: load_single_voxel(False))   # SP, TS, SP controls, SP normalizations
SVm = cached("single_voxel_matched_v2", lambda: load_single_voxel(True))
stats_sv = fig3_single_voxel(SV)

## Figures 4 and 5 — population and RDM drift, SP vs TS (all voxels)

In [ ]:
P = load_population(False)                                   # SP (raw) and TS k1 (exclAbs100)
stats_pop = fig4_population(P)
stats_rdm = fig5_rdm(P)

## Figure 6 — SP controls (single voxel)
The background version includes only subjects whose session-wise regression exists (n in the legend).

In [ ]:
stats_ctrl = fig6_sp_controls(SV, SVm)

## Figure 7 — SP response normalization

In [ ]:
Pn = {}
for z, lab in [("", "SP"), ("_zeroMean", "SP zeroMean"), ("_equalStd", "SP equalStd")]:
    Pn.update(load_population(False, models=("SP",), sp_zstr=z, label=lab))
stats_norm = fig7_normalization(SV, Pn)

## Figure S2 — SP population/RDM, all vs matched voxel count

In [ ]:
Pm_SP = load_population(True, models=("SP",))
stats_s2 = figS2_sp_matched(P, Pm_SP)